# Testing full healthcare analytics flow

This notebook traces one user question from start to finish so we can debug the full pipeline:

- schema/context discovery
- request building
- SQL generation
- SQL validation
- SQL execution
- DataFrame analysis
- chart selection and rendering

Use this as a guided debug script for one supported question, starting with:

> Show admissions by year

In [11]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

#print(f'Project root: {project_root}')

In [12]:
from src.analysis.analyzer import DataAnalyzer
from src.assistant.context import ContextBuilder
from src.assistant.request_builder import QueryRequestBuilder
from src.db.schema import SchemaManager
from src.db.sqlite_database import SQLiteDatabase
from src.sql.executor import SQLExecutor
from src.sql.generator import SQLGenerator
from src.sql.validator import SQLValidator
from src.visualization.visualizer import DataVisualizer

DB_PATH = project_root / 'data' / 'tinyehr_mimic_format.db'
QUESTION = 'Show admissions by year'

#print('DB_PATH =', DB_PATH)
#print('QUESTION =', QUESTION)

## 1) Build database context for the question

The context builder finds tables and columns relevant to the user input.
It does not execute any SQL yet; it narrows the schema to the parts we need.

In [3]:
context_builder = ContextBuilder.from_sqlite(DB_PATH)
context = context_builder.build(QUESTION)

print('Relevant tables:')
for table in context.relevant_tables:
    print(f'- {table.name}: {len(table.columns)} columns')
    print('  ', [column.name for column in table.columns[:8]])

Relevant tables:
- admissions: 16 columns
   ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location']
- date_offsets: 3 columns
   ['original_anchor_year', 'real_year', 'anchor_year_group']
- patients: 2 columns
   ['anchor_year', 'anchor_year_group']
- poe: 1 columns
   ['discontinued_by_poe_id']


## 2) Build the SQL request

This is the natural-language-to-query layer. It turns the user question into a structured request like:

- base table
- selected columns
- aggregation
- grouping
- limit

In [4]:
request_builder = QueryRequestBuilder(default_limit=100)
request = request_builder.build(QUESTION, context)
print(request)

{'base_table': 'date_offsets', 'select': [{'table': 'date_offsets', 'column': 'original_anchor_year'}, {'table': 'date_offsets', 'column': 'original_anchor_year', 'aggregate': 'COUNT', 'alias': 'record_count'}], 'group_by': [{'table': 'date_offsets', 'column': 'original_anchor_year'}], 'limit': 100}


## 3) Generate SQL

The request is converted into SQL without executing anything yet.

In [5]:
database = SQLiteDatabase(DB_PATH)
schema_manager = SchemaManager(database)

generator = SQLGenerator(schema_manager)
sql, parameters = generator.generate(request)
print('Generated SQL:')
print(sql)
print('Parameters:', parameters)

Generated SQL:
SELECT
    "date_offsets"."original_anchor_year",
    COUNT("date_offsets"."original_anchor_year") AS "record_count"
FROM "date_offsets"
GROUP BY "date_offsets"."original_anchor_year"
LIMIT 100
Parameters: []


## 4) Validate the SQL

This checks that the SQL is read-only, uses valid tables/columns, and respects the project constraints.

In [6]:
validator = SQLValidator(schema_manager)
validation = validator.validate(sql)
print('Valid?', validation.is_valid)
if not validation.is_valid:
    for issue in validation.issues:
        print(issue.code, issue.message)

Valid? True


## 5) Execute the SQL against SQLite

If validation passes, the query result becomes a pandas DataFrame.

In [7]:
executor = SQLExecutor(database, validator, max_rows=500)
query_result = executor.execute(sql, parameters)
print('SQL executed successfully.')
print('Execution time (ms):', round(query_result.execution_time_ms, 2))
print('Columns:', query_result.columns)
print(query_result.data.head())
print('\nData types:')
print(query_result.data.dtypes)

SQL executed successfully.
Execution time (ms): 2.09
Columns: ('original_anchor_year', 'record_count')
  original_anchor_year  record_count
0                 2110             2
1                 2111             2
2                 2112             1
3                 2113             3
4                 2114             2

Data types:
original_anchor_year      str
record_count            int64
dtype: object


## 6) Analyze the result

Once we have the DataFrame, we can run the analysis pipeline and inspect generated insights.

In [8]:
analyzer = DataAnalyzer()
analysis_result = analyzer.analyze(query_result.data)

print('DataFrame summary:')
print(analysis_result.dataframe_summary)
print('\nColumn statistics:')
for item in analysis_result.statistics:
    print(item)
print('\nInsights:')
for insight in analysis_result.insights:
    print(f'- {insight.title}: {insight.description}')

DataFrame summary:
DataProfile(row_count=62, column_count=2, duplicate_count=0, memory_usage_bytes=1372, missing_percentage=0.0, data_types=mappingproxy({'original_anchor_year': 'str', 'record_count': 'int64'}), high_cardinality_columns=('original_anchor_year',), constant_columns=())

Column statistics:
ColumnStatistics(column='original_anchor_year', data_type='str', minimum=None, maximum=None, mean=None, standard_deviation=None, median=None, missing_count=0)
ColumnStatistics(column='record_count', data_type='int64', minimum=1.0, maximum=4.0, mean=1.6129032258064515, standard_deviation=0.7757563989302486, median=1.0, missing_count=0)

Insights:
- Average record count: Average record count is 1.61.
- Record count is skewed: Record count has strong right skew (1.24).


## 7) Visualize the result

The visualizer selects a chart type based on the DataFrame shape and renders a Matplotlib figure.

In [9]:
visualizer = DataVisualizer()
chart_results = visualizer.visualize(query_result.data, analysis_result)
print('Number of generated charts:', len(chart_results))

for index, result in enumerate(chart_results, start=1):
    spec = result.specification
    figure = result.figure
    print(f'Chart {index}:')
    print('  title =', spec.title)
    print('  chart_type =', spec.chart_type)
    print('  x_label =', spec.x_label)
    print('  y_label =', spec.y_label)
    print('  data keys =', list(spec.data.keys()))
    print('  first row =', spec.data)

    # Render via Matplotlib in the notebook if available.
    # This is the same output used in the Streamlit app.
    display(figure)

Number of generated charts: 1
Chart 1:
  title = Admissions over Year
  chart_type = line
  x_label = Year
  y_label = Admissions
  data keys = ['original_anchor_year', 'record_count']
  first row = {'original_anchor_year': ['2110', '2111', '2112', '2113', '2114', '2115', '2116', '2117', '2118', '2119', '2120', '2123', '2125', '2128', '2129', '2131', '2132', '2133', '2134', '2136', '2137', '2138', '2140', '2141', '2142', '2143', '2144', '2145', '2146', '2147', '2148', '2150', '2153', '2154', '2155', '2157', '2160', '2161', '2162', '2166', '2167', '2169', '2170', '2172', '2174', '2175', '2176', '2177', '2178', '2180', '2181', '2182', '2183', '2184', '2185', '2186', '2187', '2189', '2193', '2196', '2200', '2201'], 'record_count': [2, 2, 1, 3, 2, 3, 1, 1, 1, 1, 2, 1, 3, 1, 3, 2, 1, 1, 2, 2, 4, 1, 2, 2, 2, 2, 1, 1, 4, 1, 2, 1, 2, 2, 2, 2, 1, 2, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 3, 1, 2, 2, 1, 1, 1, 1]}


<Figure size 800x500 with 1 Axes>